In [9]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import requests
from bs4 import BeautifulSoup



In [10]:

def geocode_address_census(address):
    """
    Geocode a single address using the Census Geocoder and return lat/lon.
    
    Args:
        address (str): The address to geocode (e.g., "501 Canyon Ridge Dr, Austin, TX 78753")
    
    Returns:
        dict: Dictionary with 'lat', 'lon', and 'matched_address' or None if not found
    """
    # Base URL for the Census Geocoder
    base_url = "https://geocoding.geo.census.gov/geocoder/locations/onelineaddress"
    
    # Parameters for the request
    params = {
        'address': address,
        'benchmark': '4'  # Public_AR_Current benchmark
    }
    
    try:
        # Send GET request
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        
        # Parse HTML with BeautifulSoup
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Find the div containing the results
        # Based on your HTML, the results are in a div with line breaks and <b> tags
        result_div = soup.find('div')
        if not result_div:
            return None
            
        # Initialize variables
        lat = None
        lon = None
        matched_address = None
        
        # Find all <b> tags which contain the labels
        bold_tags = result_div.find_all('b')
        
        for i, bold in enumerate(bold_tags):
            label = bold.get_text(strip=True)
            
            # Look for Interpolated Latitude
            if 'Latitude' in label or 'latitude' in label:
                # Latitude value is in the following <span>
                lat_span = bold.find_next('span')
                if lat_span:
                    lat = float(lat_span.get_text(strip=True))
            
            # Look for Interpolated Longitude
            elif 'Longitude' in label or 'longitude' in label:
                lon_span = bold.find_next('span')
                if lon_span:
                    lon = float(lon_span.get_text(strip=True))
            
            # Look for Matched Address
            elif 'Matched Address' in label or 'matched address' in label:
                matched_span = bold.find_next('span')
                if matched_span:
                    matched_address = matched_span.get_text(strip=True)
        
        # Alternative: Directly find the spans by searching for text
        if not lat or not lon:
            # Find all spans that might contain coordinates
            all_spans = result_div.find_all('span')
            for span in all_spans:
                text = span.get_text(strip=True)
                # Check if this span contains a coordinate (looks like a decimal number)
                if text and text[0] in '-0123456789' and '.' in text:
                    # Check if it's the longitude (usually more negative for US addresses)
                    if text.startswith('-') and abs(float(text)) > 70:  # Approximate longitude range
                        lon = float(text)
                    elif abs(float(text)) < 50:  # Approximate latitude range for US
                        lat = float(text)
        
        if lat is not None and lon is not None:
            return {
                'matched_address': matched_address,
                'latitude': lat,
                'longitude': lon
            }
        else:
            return None
            
    except requests.exceptions.RequestException as e:
        print(f"Error making request: {e}")
        return None
    except Exception as e:
        print(f"Error parsing response: {e}")
        return None

# Example usage with the address you provided
address = "501 Canyon Ridge Dr, Austin, TX 78753"
result = geocode_address_census(address)

if result:
    print(f"Matched Address: {result['matched_address']}")
    print(f"Latitude: {result['latitude']}")
    print(f"Longitude: {result['longitude']}")
else:
    print("Geocoding failed")

Matched Address: 501 CANYON RIDGE DR W, AUSTIN, TX, 78753
Latitude: 30.402743967966
Longitude: -97.67121877523


In [11]:


def scrape_yellowpages_with_coordinates(page_number):
    """
    Scrape Yellow Pages and geocode each address
    """
    # [Your Selenium setup code here - same as before]
    
    # Configure Chrome options
    options = webdriver.ChromeOptions()
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    url = f'https://www.yellowpages.com/austin-tx/restaurants?page={page_number}'
    page_data = []
    
    try:
        print(f"\n--- Scraping Page {page_number} ---")
        driver.get(url)
        time.sleep(5)
        
        listings = driver.find_elements(By.CLASS_NAME, "result")
        print(f"Found {len(listings)} listings on page {page_number}")
        
        for i, listing in enumerate(listings, 1):
            try:
                info_div = listing.find_element(By.CLASS_NAME, "info")
                
                # Extract business name and categories (as before)
                name_element = info_div.find_element(By.CLASS_NAME, "business-name")
                business_name = name_element.text.strip()
                
                # Extract categories
                categories = []
                try:
                    categories_div = info_div.find_element(By.CLASS_NAME, "categories")
                    category_links = categories_div.find_elements(By.TAG_NAME, "a")
                    categories = [cat.text.strip() for cat in category_links]
                except:
                    pass
                
                # Extract address
                address = "N/A"
                try:
                    adr_div = info_div.find_element(By.CLASS_NAME, "adr")
                    street = adr_div.find_element(By.CLASS_NAME, "street-address").text.strip()
                    locality = adr_div.find_element(By.CLASS_NAME, "locality").text.strip()
                    address = f"{street}, {locality}"
                except:
                    try:
                        adr_div = info_div.find_element(By.CLASS_NAME, "adr")
                        address = adr_div.text.strip().replace('\n', ', ')
                    except:
                        pass
                
                # Geocode the address using Census API
                coordinates = None
                if address != "N/A":
                    coordinates = geocode_address_census(address)
                    # Be respectful to the Census API
                    time.sleep(1)
                
                # Add to page data
                record = {
                    'page': page_number,
                    'name': business_name,
                    'categories': ', '.join(categories),
                    'address': address,
                    'matched_address': coordinates['matched_address'] if coordinates else None,
                    'latitude': coordinates['latitude'] if coordinates else None,
                    'longitude': coordinates['longitude'] if coordinates else None
                }
                page_data.append(record)
                
                print(f"  ✓ {i}. {business_name}")
                if coordinates:
                    print(f"     → Coordinates: {coordinates['latitude']}, {coordinates['longitude']}")
                
            except Exception as e:
                print(f"  ✗ Error on listing {i}")
                continue
                
    except Exception as e:
        print(f"Error on page {page_number}: {e}")
        
    finally:
        driver.quit()
        
    return page_data

# Scrape one page with geocoding
results = scrape_yellowpages_with_coordinates(2)

# Save to CSV
df = pd.DataFrame(results)
df.to_csv('yellowpages_with_coordinates.csv', index=False)
print(f"\n✅ Saved {len(results)} listings with coordinates to CSV")


--- Scraping Page 2 ---
Found 30 listings on page 2
  ✓ 1. Pei Wei
     → Coordinates: 30.298710536593, -97.720197949461
  ✓ 2. The Captain's Seafood & Oyster Bar
  ✓ 3. Julio's Restaurant
     → Coordinates: 30.304160528115, -97.726501339235
  ✓ 4. El Faro Taco
  ✓ 5. Mimi's Cafe
     → Coordinates: 30.306791676578, -97.937682453327
  ✓ 6. Los Pinos Mexican Restaurant
  ✓ 7. Dr Wok
     → Coordinates: 30.444978538457, -97.777894104141
  ✓ 8. Roaring Fork Stonelake
     → Coordinates: 30.40008604809, -97.735861630014
  ✓ 9. Niki's Pizza
     → Coordinates: 30.416825968424, -97.669212112437
  ✓ 10. Restaurant
     → Coordinates: 30.259731669478, -97.738360024825
  ✓ 11. Sushi Sake Japanese Cuisine
     → Coordinates: 30.386486222049, -97.74256076389
  ✓ 12. French Quarter Grille
  ✓ 13. Enchiladas Y Mas
     → Coordinates: 30.353852756177, -97.726188016192
  ✓ 14. 24 Diner
     → Coordinates: 30.271614976785, -97.75399983192
  ✓ 15. Freda's Seafood Grille
     → Coordinates: 30.4647730